In [14]:
import pandas as pd

import optuna
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

from src.data import load_csv_df
from src.plotting import set_plotting_style

In [8]:
# Already prepared accent palette applying using function from mine src module (plotting)
set_plotting_style()
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [9]:
DF_PATH = "../data/processed/pca_df.csv"

# Safely loading dataframe using function from mine src module (data)
X_pca_df = load_csv_df(
    df_path=DF_PATH,
)

Successfully loaded dataframe: ..\data\processed\pca_df.csv
Dataframe has: 195 rows / 3 columns
Dataframe size: 0.005MB


In [15]:
"""Comparative clustering optimization (optuna)
-----------------------------------------------------
1. Objective & metrics:
    - Optimize hyperparameters for comparative clustering (K-means, DBSCAN, HDBSCAN).
    - Maximize Silhouette score as primary metric to ensure groups.
    - Track Davies-Bouldin and Calinski-Harabasz scores for quantitative validation.

2. Quality & validation:
    - Minimal cluster size (n >= 3): Reject micro-clusters with < 3 countries to
    avoid isolating single outlier nations.
    - Max noise threshold (<= 20%): Limit unclassified noise points in DBSCAN/HDBSCAN
    so we don't lose too many countries from analysis.

3. Experiment tracking:
    - Log all hyperparameter combinations and trial statuses via trial.set_user_attr().
    - Export results to Pandas DataFrames for quick cross-algorithm comparisons.
"""

def objective_kmeans(trial: optuna.Trial) -> float:
    n_clusters = trial.suggest_int("n_clusters", 2, 10)
    init = trial.suggest_categorical("init", ["random", "k-means++"])
    n_init = trial.suggest_int("n_init", 10, 20)
    algorithm = trial.suggest_categorical("algorithm", ["lloyd", "elkan"])

    model = KMeans(
        n_clusters=n_clusters,
        init=init,
        n_init=n_init,
        algorithm=algorithm,
    )
    labels = model.fit_predict(X_pca_df)

    cluster_sizes = pd.Series(labels).value_counts()
    min_cluster_size = int(cluster_sizes.min())
    max_cluster_size = int(cluster_sizes.max())

    trial.set_user_attr("min_cluster_size", min_cluster_size)
    trial.set_user_attr("max_cluster_size", max_cluster_size)

    if min_cluster_size < 3:
        trial.set_user_attr("status", "Rejected (<3 members)")
        return -1

    score = silhouette_score(X_pca_df, labels)
    db_score = davies_bouldin_score(X_pca_df, labels)
    ch_score = calinski_harabasz_score(X_pca_df, labels)

    trial.set_user_attr("davies_bouldin", db_score)
    trial.set_user_attr("calinski_harabasz", ch_score)
    trial.set_user_attr("status", "Accepted")

    return score


study_kmeans = optuna.create_study(direction="maximize")
study_kmeans.optimize(
    objective_kmeans,
    n_trials=50,
    show_progress_bar=True
)

print("=" * 50)
print(f"Best silhouette score: {study_kmeans.best_value:.4f}")
print("Best hyperparameters:")
for key, value in study_kmeans.best_params.items():
    print(f"  - {key}: {value}")
print("=" * 50)

df_kmeans_results = study_kmeans.trials_dataframe()

cols = [
    "number", "value", "params_n_clusters", "params_init",
    "params_algorithm", "user_attrs_davies_bouldin",
    "user_attrs_min_cluster_size", "user_attrs_status"
]

df_kmeans_summary = df_kmeans_results[cols].sort_values(by="value", ascending=False).reset_index(drop=True)
display(df_kmeans_summary.head(10))


best_kmeans = KMeans(
    **study_kmeans.best_params,
    random_state=42
)
X_pca_df["kmeans_cluster"] = best_kmeans.fit_predict(X_pca_df)

Best trial: 9. Best value: 0.387798: 100%|██████████| 50/50 [00:02<00:00, 22.05it/s]

Best silhouette score: 0.3878
Best hyperparameters:
  - n_clusters: 2
  - init: random
  - n_init: 12
  - algorithm: lloyd


,number,value,params_n_clusters,params_init,params_algorithm,user_attrs_davies_bouldin,user_attrs_min_cluster_size,user_attrs_status
0,17,0.387798,2,k-means++,lloyd,0.959597,37,Accepted
1,9,0.387798,2,random,lloyd,0.959597,37,Accepted
2,14,0.387798,2,random,lloyd,0.959597,37,Accepted
3,36,0.387798,2,random,lloyd,0.959597,37,Accepted
4,43,0.387798,2,k-means++,lloyd,0.959597,37,Accepted
5,42,0.387798,2,k-means++,lloyd,0.959597,37,Accepted
6,28,0.387798,2,random,lloyd,0.959597,37,Accepted
7,32,0.387798,2,k-means++,elkan,0.959597,37,Accepted
8,47,0.387798,2,random,lloyd,0.959597,37,Accepted
9,48,0.387798,2,k-means++,lloyd,0.959597,37,Accepted
